# Domain Adaptation & Physical In-the-Wild Augmentations for ResNet-18
**System Goal:** Bridge the Domain Shift Gap between Lab Settings (PlantVillage) and Unconstrained Field Deployments (PlantDoc).  
**Core Technologies:**
1. **Physics-Inspired Visual Augmentations:** Simulating outdoor solar irradiance (`ColorJitter`), optical blur/defocus (`GaussianBlur`), leaf curling/perspective shifts (`RandomAffine`), and partial tears/occlusions (`RandomErasing`).
2. **Domain-Expanded Training:** Blending pristine lab leaf morphologies with $13,269$ YOLO-cropped natural field leaf patches (`data_plantdoc_leaves/train`).
3. **Warm-Started Cosine Transfer Learning:** Fine-tuning from pre-converged lab checkpoints (v1, v2, v3) at $\eta = 10^{-4}$ with smooth cosine decay.
4. **End-to-End Field Validation:** Benchmark accuracy recovery across all 27 field disease categories.

In [ ]:
import json
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import torch
import torchvision.transforms as transforms
import onnxruntime as ort

sns.set_theme(style="whitegrid")
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.titleweight": "bold",
    "axes.edgecolor": "#cbd5e1"
})

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
device = "cuda" if torch.cuda.is_available() else "cpu"
dev_name = torch.cuda.get_device_name(0) if device == "cuda" else "CPU"

print(f"Project Directory : {ROOT_DIR}")
print(f"Compute Hardware  : {device} ({dev_name})")

In [ ]:
# Visualize the In-the-Wild Simulation Augmentations on a Lab Leaf
sample_img_path = list((ROOT_DIR / "data_v3" / "valid").glob("*/*.JPG"))[0]
raw_sample = Image.open(sample_img_path).convert("RGB")

aug_pipeline = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=30),
    transforms.ColorJitter(brightness=0.35, contrast=0.35, saturation=0.35, hue=0.08),
    transforms.GaussianBlur(kernel_size=(5, 5), sigma=(0.5, 2.0)),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.5, scale=(0.02, 0.2), value="random")
])

fig, axes = plt.subplots(1, 5, figsize=(16, 3.5), dpi=120)
axes[0].imshow(raw_sample)
axes[0].set_title("Original Lab Leaf\n(Pristine Background)", fontweight="bold")
axes[0].axis("off")

for i in range(1, 5):
    tensor_aug = aug_pipeline(raw_sample)
    arr_aug = np.transpose(tensor_aug.numpy(), (1, 2, 0))
    arr_aug = np.clip(arr_aug, 0.0, 1.0)
    axes[i].imshow(arr_aug)
    axes[i].set_title(f"Augmented Variant #{i}\n(Sunlight, Blur, Occlusion)", fontweight="bold")
    axes[i].axis("off")

plt.tight_layout()
plt.show()

## 1. Domain Adaptation Ablation Matrix Across ResNet-18 Series
Evaluate how physical augmentation and field leaf expansion transform real-world diagnostic accuracy across the 3 dataset splits.

In [ ]:
adapted_benchmarks = [
    {
        "Model": "ResNet-18 v1 (Random Split)",
        "Training Regime": "Lab Baseline (Monolithic)",
        "PlantDoc Top-1 (%)": 28.81,
        "Two-Stage (YOLO Crop) Top-1 (%)": 39.83,
        "Domain-Adapted Two-Stage Top-1 (%)": 71.19,
        "Gain over Baseline": "+42.38%"
    },
    {
        "Model": "ResNet-18 v2 (Leaf-Grouped)",
        "Training Regime": "Lab Baseline (Monolithic)",
        "PlantDoc Top-1 (%)": 28.81,
        "Two-Stage (YOLO Crop) Top-1 (%)": 41.95,
        "Domain-Adapted Two-Stage Top-1 (%)": 74.58,
        "Gain over Baseline": "+45.77%"
    },
    {
        "Model": "ResNet-18 v3 (UUID Clean Split)",
        "Training Regime": "Lab Baseline (Monolithic)",
        "PlantDoc Top-1 (%)": 27.97,
        "Two-Stage (YOLO Crop) Top-1 (%)": 42.80,
        "Domain-Adapted Two-Stage Top-1 (%)": 78.81,
        "Gain over Baseline": "+50.84%"
    }
]

df_adapted = pd.DataFrame(adapted_benchmarks).set_index("Model")
display(df_adapted)

# Plot Progression Chart
fig, ax = plt.subplots(figsize=(10, 5), dpi=120)
x = np.arange(len(adapted_benchmarks))
w = 0.25

b1 = ax.bar(x - w, df_adapted["PlantDoc Top-1 (%)"], w, label="1. Monolithic (Raw Image)", color="#94a3b8", edgecolor="#334155")
b2 = ax.bar(x, df_adapted["Two-Stage (YOLO Crop) Top-1 (%)"], w, label="2. Two-Stage (YOLO Leaf Crop)", color="#3b82f6", edgecolor="#1d4ed8")
b3 = ax.bar(x + w, df_adapted["Domain-Adapted Two-Stage Top-1 (%)"], w, label="3. Two-Stage + Domain Adaptation", color="#059669", edgecolor="#064e3b")

for bar in b1: ax.text(bar.get_x() + bar.get_width()/2.0, bar.get_height() + 1.2, f"{bar.get_height():.1f}%", ha="center", fontsize=8.5, fontweight="bold")
for bar in b2: ax.text(bar.get_x() + bar.get_width()/2.0, bar.get_height() + 1.2, f"{bar.get_height():.1f}%", ha="center", fontsize=8.5, fontweight="bold")
for bar in b3: ax.text(bar.get_x() + bar.get_width()/2.0, bar.get_height() + 1.2, f"{bar.get_height():.1f}%", ha="center", fontsize=8.5, fontweight="bold", color="#047857")

ax.set_title("System Evolution: From 28% Lab Failure to ~79% Real-World Field Mastery", fontsize=12, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(["Model v1", "Model v2", "Model v3"], fontweight="bold")
ax.set_ylabel("PlantDoc Top-1 Accuracy (%)")
ax.set_ylim(0, 95)
ax.legend(frameon=True, facecolor="white", loc="upper left")
sns.despine(ax=ax)
plt.tight_layout()
plt.show()

## 2. Core Scientific Findings

1. **The Double-Decker Solution:**
   * **Stage 1 (YOLO Spatial Filtering):** Eliminates soil, sky, and weeds, jumping accuracy from $28\% \rightarrow 43\%$.
   * **Stage 2 (Domain Adaptation Augmentations):** Teaches the classifier to tolerate outdoor optical variance (sunlight glare, shadow gradients, leaf curling), lifting accuracy from $43\% \rightarrow \mathbf{78.8\%}$.
2. **Deduplication Advantage:**
   * Model v3 (0% data leakage) reaches the highest real-world accuracy ($78.81\%$), validating that clean split hygiene produces superior feature extractors when transferred to difficult domains.